# Structured I/O, Schema Validation & JSON Pipelines

In production AI systems, LLM outputs must interface reliably with downstream databases, API endpoints, microservices, and UI clients. Unstructured natural language responses can break downstream parsers.

### Key Objectives:
1. Define robust schemas using **Pydantic v2** (`BaseModel`, `Field`) and **TypedDict**.
2. Enforce schema guarantees using `model.with_structured_output(...)`.
3. Handle complex nested models, lists, and enumerations.
4. Serialize and deserialize schemas with `.model_dump()`, `.model_dump_json()`, and `json.dumps()`.
5. Build an automated end-to-end file ingestion and entity extraction pipeline using the `docs/` directory.

## 1. Environment Setup

Initialize the model, verify the `docs/` storage directory, and load environment configuration.

In [ ]:
import os
import json
from enum import Enum
from typing import List, Optional
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# Ensure docs directory exists
docs_dir = Path("docs")
docs_dir.mkdir(parents=True, exist_ok=True)

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0 # Strict determinism for structured extraction
)
print("Model Initialized for Structured Output:", model.model)
print("Docs Directory:", docs_dir.resolve())

## 2. Defining Structured Schemas with Pydantic & Enums

We define structured domain entities using Pydantic `BaseModel`. The `Field` description provides semantic clues directly to the LLM's function/tool calling engine.

In [ ]:
class Currency(str, Enum):
    USD = "USD"
    EUR = "EUR"
    GBP = "GBP"
    NGN = "NGN"

class LineItem(BaseModel):
    description: str = Field(..., description="Name or description of the purchased good or service")
    quantity: int = Field(default=1, description="Number of units purchased")
    unit_price: float = Field(..., description="Price per unit")
    total_price: float = Field(..., description="Total line amount (quantity * unit_price)")

class VendorInfo(BaseModel):
    name: str = Field(..., description="Name of the vendor or issuing business")
    tax_id: Optional[str] = Field(None, description="Tax ID or VAT registration number if available")
    contact_email: Optional[str] = Field(None, description="Support or billing email address")

class InvoiceExtraction(BaseModel):
    """Complete parsed invoice schema"""
    invoice_number: str = Field(..., description="Unique invoice or reference identifier")
    invoice_date: str = Field(..., description="Issue date in YYYY-MM-DD format")
    vendor: VendorInfo = Field(..., description="Information regarding the issuing vendor")
    items: List[LineItem] = Field(default_factory=list, description="All individual itemized lines")
    subtotal: float = Field(..., description="Subtotal amount before taxes and discounts")
    tax_amount: float = Field(default=0.0, description="Tax or VAT total amount")
    total_amount: float = Field(..., description="Final balance due")
    currency: Currency = Field(..., description="Three-letter currency code")

print("Invoice Schema defined successfully with", len(InvoiceExtraction.model_fields), "fields.")

## 3. Enforcing Structured Responses with `.with_structured_output()`

When calling `model.with_structured_output(InvoiceExtraction)`, LangChain binds the JSON schema to the model's native function/tool-calling interface, guaranteeing typed Pydantic objects on invocation.

In [ ]:
# Bind the schema to the model
structured_extractor = model.with_structured_output(InvoiceExtraction)

# Ingest raw invoice text from docs/sample_invoice.txt
invoice_file = Path("docs/sample_invoice.txt")
raw_invoice_text = invoice_file.read_text(encoding="utf-8")

parsed_invoice = structured_extractor.invoke(raw_invoice_text)
print("Parsed Object Type:", type(parsed_invoice))
print("Invoice Number:", parsed_invoice.invoice_number)
print("Vendor Name:", parsed_invoice.vendor.name)
print("Total Items:", len(parsed_invoice.items))
for item in parsed_invoice.items:
    print(f" - {item.description}: {item.quantity} x ${item.unit_price} = ${item.total_price}")
print(f"Total Due: {parsed_invoice.total_amount} {parsed_invoice.currency.value}")

## 4. JSON Serialization & Deserialization

Pydantic v2 offers fast native serialization:
* `parsed_invoice.model_dump()`: Converts into a standard Python `dict`.
* `parsed_invoice.model_dump_json(indent=2)`: Serializes directly into a formatted JSON string.
* `InvoiceExtraction.model_validate_json(...)`: Parses and validates raw JSON strings back into strongly-typed objects.

In [ ]:
# Convert to Dict
dict_payload = parsed_invoice.model_dump()
print("Python Dict representation keys:", list(dict_payload.keys()))

# Serialize to JSON string with indentation
json_str = parsed_invoice.model_dump_json(indent=2)
print("\nFormatted JSON String:\n", json_str)

# Validate & deserialize back from JSON string
reconstructed_invoice = InvoiceExtraction.model_validate_json(json_str)
assert reconstructed_invoice.invoice_number == parsed_invoice.invoice_number
print("\nDeserialization and validation check passed successfully!")

## 5. End-to-End Pipeline: Ingesting Files from `docs/`, Extracting Schema & Saving to `docs/`

Let's build a real-world pipeline that reads a candidate profile from `docs/candidate_resume.txt`, extracts structured resume data, and writes a validated JSON record to `docs/candidate_profile.json`.

In [ ]:
class SeniorityLevel(str, Enum):
    JUNIOR = "Junior"
    MID = "Mid-Level"
    SENIOR = "Senior"
    STAFF_OR_LEAD = "Staff/Lead"

class CandidateProfile(BaseModel):
    full_name: str = Field(..., description="Full legal name of candidate")
    email: Optional[str] = Field(None, description="Contact email")
    primary_skills: List[str] = Field(..., description="Top 5 core technical competencies")
    years_of_experience: int = Field(..., description="Total estimated years in software/data engineering")
    seniority: SeniorityLevel = Field(..., description="Evaluated seniority bracket")
    summary: str = Field(..., description="Two-sentence executive candidate evaluation")

# Step 1: Read the resume text file stored in docs/
resume_file = Path("docs/candidate_resume.txt")
resume_content = resume_file.read_text(encoding="utf-8")
print(f"Ingested {len(resume_content)} bytes from: {resume_file}")

# Step 2: Parse through structured model
candidate_extractor = model.with_structured_output(CandidateProfile)
candidate_data = candidate_extractor.invoke(resume_content)

# Step 3: Write structured JSON output back to docs/
output_json_path = Path("docs/candidate_profile.json")
output_json_path.write_text(candidate_data.model_dump_json(indent=2), encoding="utf-8")

print(f"Extraction Pipeline Complete! Output written to: {output_json_path}")
print("\nParsed Content Preview:")
print(candidate_data.model_dump_json(indent=2))